> **Traçabilité (fusion Codes_finaux) :** notebook présent uniquement dans `Codes prof/spam_5_DNN_Keras_Optuna.ipynb` (pas d'équivalent Colab), copié tel quel.

---
# DNN : `spam` (avec `Keras` et `Optuna`)
---

## Packages

In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from keras import callbacks, layers, models

import optuna
from optuna.visualization import plot_optimization_history, plot_param_importances

---
## 1. Données
---

### 1.1. Importation

In [2]:
# En local :
directory = '/Users/vincentlefieux/Dropbox/Docs_ACADEMIQUE/Data/'

# Sur Google collab ou Onyxia (sur un répertoire temporaire) :
# directory = ''

# Sur Google collab (sur le drive) :
# from google.colab import drive
# drive.mount('/content/drive')
# directory = '/content/drive/MyDrive/Data/'

In [3]:
data = pd.read_csv(directory + 'spam.csv',
                   header    = 0,
                   index_col = None,
                   sep       = ',',
                   decimal   = '.')

In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4601 entries, 0 to 4600
Data columns (total 58 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   word_freq_make              4601 non-null   float64
 1   word_freq_address           4601 non-null   float64
 2   word_freq_all               4601 non-null   float64
 3   word_freq_3d                4601 non-null   float64
 4   word_freq_our               4601 non-null   float64
 5   word_freq_over              4601 non-null   float64
 6   word_freq_remove            4601 non-null   float64
 7   word_freq_internet          4601 non-null   float64
 8   word_freq_order             4601 non-null   float64
 9   word_freq_mail              4601 non-null   float64
 10  word_freq_receive           4601 non-null   float64
 11  word_freq_will              4601 non-null   float64
 12  word_freq_people            4601 non-null   float64
 13  word_freq_report            4601 

In [5]:
data.head()

,word_freq_make,word_freq_address,word_freq_all,word_freq_3d,word_freq_our,word_freq_over,word_freq_remove,word_freq_internet,word_freq_order,word_freq_mail,...,char_freq_semicolon,char_freq_leftbrac,char_freq_leftsquarebrac,char_freq_exclaim,char_freq_dollar,char_freq_pound,capital_run_length_average,capital_run_length_longest,capital_run_length_total,spam
0,0.00,0.64,0.64,0.0,0.32,0.00,0.00,0.00,0.00,0.00,...,0.00,0.000,0.0,0.778,0.000,0.000,3.756,61,278,1
1,0.21,0.28,0.50,0.0,0.14,0.28,0.21,0.07,0.00,0.94,...,0.00,0.132,0.0,0.372,0.180,0.048,5.114,101,1028,1
2,0.06,0.00,0.71,0.0,1.23,0.19,0.19,0.12,0.64,0.25,...,0.01,0.143,0.0,0.276,0.184,0.010,9.821,485,2259,1
3,0.00,0.00,0.00,0.0,0.63,0.00,0.31,0.63,0.31,0.63,...,0.00,0.137,0.0,0.137,0.000,0.000,3.537,40,191,1
4,0.00,0.00,0.00,0.0,0.63,0.00,0.31,0.63,0.31,0.63,...,0.00,0.135,0.0,0.135,0.000,0.000,3.537,40,191,1


### 1.2. Gestion des données manquantes

In [6]:
missing_percentage = data.isna().mean() * 100

print('MISSING VALUES :')
if missing_percentage[missing_percentage != 0].empty:
    print('No')
else:
    print(missing_percentage[missing_percentage != 0].sort_values(ascending=False))

MISSING VALUES :
No


### 1.3. Gestion des variables

In [7]:
target = 'spam'

y = data[target]
X = data.drop(target, axis=1)

### 1.4. Création des échantillons de validation et test

In [8]:
test_portion  = 1/5
valid_portion = 1/5

X_train_valid, X_test, y_train_valid, y_test = train_test_split(X, y, test_size=test_portion, random_state=1234)

X_train, X_valid, y_train, y_valid = train_test_split(X_train_valid, y_train_valid, test_size=valid_portion)

print('Dimensions de X_train :', X_train.shape)
print('Dimensions de X_valid :', X_valid.shape)
print('Dimensions de X_test  :', X_test.shape)

print('Dimensions de y_train :', y_train.shape)
print('Dimensions de y_valid :', y_valid.shape)
print('Dimensions de y_test  :', y_test.shape)

Dimensions de X_train : (2944, 57)
Dimensions de X_valid : (736, 57)
Dimensions de X_test  : (921, 57)
Dimensions de y_train : (2944,)
Dimensions de y_valid : (736,)
Dimensions de y_test  : (921,)


### 1.5. Normalisation des covariables

On normalise (centrage-réduction) les covariables :

In [9]:
scaler = StandardScaler(with_mean=True, with_std=True)
scaler.fit(X_train)

X_train_norm = scaler.transform(X_train)
X_valid_norm = scaler.transform(X_valid)
X_test_norm  = scaler.transform(X_test)

---
## 2. DNN
---

### 2.1. Architecture et optimiseur

In [10]:
dim_inputs  = (X_train_norm.shape[1],)
dim_outputs = 1

n_layers_min = 1
n_layers_max = 3

n_units_min  = 32
n_units_max  = 128
n_units_step = 32

dropout_min  = 0.2
dropout_max  = 0.2
dropout_step = 0.1

activation_hidden = 'relu'
activation_output = 'sigmoid'

optimizer = 'adam'
loss      = 'binary_crossentropy'
metrics   = ['accuracy']

monitor  = 'val_loss'
mode     = 'min'
patience = 10

batch_size   = 300
n_epochs_max = 200
verbose      = 0

def objective(trial):
    num_layers = trial.suggest_int('num_layers', n_layers_min, n_layers_max)
    layers_units = []
    for i in range(num_layers):
        units = trial.suggest_int(f'units_layer_{i}', n_units_min, n_units_max, step=n_units_step)
        layers_units.append(units)
    dropout_rate = trial.suggest_float('dropout_rate', dropout_min, dropout_max, step=dropout_step)
    # learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)
    
    model = models.Sequential()
    for units in layers_units:
        model.add(layers.Dense(units, activation=activation_hidden))
        model.add(layers.Dropout(dropout_rate))
    model.add(layers.Dense(dim_outputs, activation=activation_output))
    
    model.compile(optimizer=optimizer,
                  # optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
                  loss=loss,
                  metrics=metrics)
    
    callback = callbacks.EarlyStopping(monitor              = monitor,
                                       mode                 = mode,
                                       patience             = patience,
                                       restore_best_weights = True)
    
    hist = model.fit(X_train_norm,
                     y_train,
                     validation_data = (X_valid_norm, y_valid),
                     batch_size      = batch_size,
                     epochs          = n_epochs_max,
                     verbose         = verbose,
                     callbacks       = [callback])
    
    score_valid = model.evaluate(X_valid_norm, y_valid, verbose=0)
    loss_valid = score_valid[0]
    
    return loss_valid

### 2.2. Entraînement

In [11]:
n_trials = 50

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=n_trials)

print('Meilleurs hyperparamètres       :', study.best_params)
print('Meilleure entropie (validation) :', study.best_value)

plot_optimization_history(study).show()
plot_param_importances(study).show()

# Pour la sauvegarde et le chargement de l'étude
# import joblib
# joblib.dump(study, 'optuna_study.pkl') # Sauvegarde
# study = joblib.load('optuna_study.pkl') # Chargement

[I 2025-12-31 18:48:21,375] A new study created in memory with name: no-name-122fa70a-5ec7-4d6e-b008-6057666172c2
[I 2025-12-31 18:48:22,541] Trial 0 finished with value: 0.1907043308019638 and parameters: {'num_layers': 3, 'units_layer_0': 96, 'units_layer_1': 64, 'units_layer_2': 96, 'dropout_rate': 0.2}. Best is trial 0 with value: 0.1907043308019638.
[I 2025-12-31 18:48:23,829] Trial 1 finished with value: 0.19267040491104126 and parameters: {'num_layers': 2, 'units_layer_0': 32, 'units_layer_1': 96, 'dropout_rate': 0.2}. Best is trial 0 with value: 0.1907043308019638.
[I 2025-12-31 18:48:25,464] Trial 2 finished with value: 0.1877075731754303 and parameters: {'num_layers': 1, 'units_layer_0': 32, 'dropout_rate': 0.2}. Best is trial 2 with value: 0.1877075731754303.
[I 2025-12-31 18:48:26,509] Trial 3 finished with value: 0.1869889348745346 and parameters: {'num_layers': 3, 'units_layer_0': 96, 'units_layer_1': 32, 'units_layer_2': 32, 'dropout_rate': 0.2}. Best is trial 3 with val

Meilleurs hyperparamètres       : {'num_layers': 1, 'units_layer_0': 96, 'dropout_rate': 0.2}
Meilleure entropie (validation) : 0.17727051675319672


### 2.3. Prévisions

In [12]:
best_params = study.best_trial.params

model = models.Sequential()
model.add(layers.Input(dim_inputs))
for i in range(best_params['num_layers']):
    units = best_params[f'units_layer_{i}']
    model.add(layers.Dense(units, activation=activation_hidden))
    model.add(layers.Dropout(best_params['dropout_rate']))
model.add(layers.Dense(dim_outputs, activation=activation_output))

model.compile(optimizer=optimizer,
              # optimizer=tf.keras.optimizers.Adam(learning_rate=best_params['learning_rate']),
              loss=loss,
              metrics=metrics)

callback = callbacks.EarlyStopping(monitor              = monitor,
                                   mode                 = mode,
                                   patience             = patience,
                                   restore_best_weights = True)

hist = model.fit(X_train_norm,
                 y_train,
                 batch_size      = batch_size,
                 validation_data = (X_valid_norm, y_valid),
                 epochs          = n_epochs_max,
                 callbacks       = [callback])

Epoch 1/200
10/10 [==============================] - 0s 6ms/step - loss: 0.6391 - accuracy: 0.6508 - val_loss: 0.5325 - val_accuracy: 0.8057
Epoch 2/200
10/10 [==============================] - 0s 2ms/step - loss: 0.4698 - accuracy: 0.8336 - val_loss: 0.4177 - val_accuracy: 0.8750
Epoch 3/200
10/10 [==============================] - 0s 2ms/step - loss: 0.3795 - accuracy: 0.8869 - val_loss: 0.3571 - val_accuracy: 0.8981
Epoch 4/200
10/10 [==============================] - 0s 2ms/step - loss: 0.3310 - accuracy: 0.8991 - val_loss: 0.3223 - val_accuracy: 0.8995
Epoch 5/200
10/10 [==============================] - 0s 2ms/step - loss: 0.3036 - accuracy: 0.9066 - val_loss: 0.2994 - val_accuracy: 0.9022
Epoch 6/200
10/10 [==============================] - 0s 2ms/step - loss: 0.2806 - accuracy: 0.9127 - val_loss: 0.2830 - val_accuracy: 0.9062
Epoch 7/200
10/10 [==============================] - 0s 2ms/step - loss: 0.2628 - accuracy: 0.9134 - val_loss: 0.2711 - val_accuracy: 0.9130
Epoch 8/200
1

In [13]:
y_test_pred = model.predict(X_test_norm)
y_test_pred[0:5]

29/29 [==============================] - 0s 314us/step


array([[9.9994951e-01],
       [3.4255937e-02],
       [4.1955666e-08],
       [3.4475876e-03],
       [2.9959751e-04]], dtype=float32)

In [14]:
y_test_pred_classes = (y_test_pred > 0.5).astype(int)
y_test_pred_classes[0:5]

array([[1],
       [0],
       [0],
       [0],
       [0]])

In [15]:
score_test = model.evaluate(X_test_norm, y_test, verbose=0)
print(f'Entropie test   : {score_test[0]:4.4f}')
print(f'Exactitude test : {score_test[1]:4.4f}')

Entropie test   : 0.1465
Exactitude test : 0.9522
